# RedConecta Telecom S. A. - Análisis operacional del servicio

**Objetivo:** comparar el desempeño por región y tipo de servicio y generar los archivos de salida para Soporte Técnico y Experiencia de Clientes.

Las funciones reutilizables están en `src/` (`carga`, `metricas`, `visualizacion`).

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))

import pandas as pd
import plotly.express as px

from carga import cargar_datos
from metricas import (comparar_grupos, ic_sla_bootstrap, independencia_sla,
                      matriz_correlacion, resumen_grupos)
from visualizacion import guardar_dashboard

## 1. Carga de datos

In [2]:
df = cargar_datos(RAIZ / 'data' / 'raw' / 'dataset_set_D_telecomunicaciones.xlsx')
print(df.shape, df.fecha_registro.min().date(), '->', df.fecha_registro.max().date())
df.head()

(200, 11) 2025-03-01 -> 2025-09-16


,fecha_registro,region,tipo_servicio,trafico_datos_gb,latencia_ms,incidencias_reportadas,tickets_soporte,tiempo_resolucion_hr,clientes_afectados,cumplimiento_sla,sla_cumple
0,2025-03-01,Centro,Empresas,457.8,21.3,0,27,22.4,144,No cumple,0
1,2025-03-02,Norte,Internet Hogar,383.4,36.5,13,19,39.4,290,No cumple,0
2,2025-03-03,Norte,Empresas,457.3,97.4,7,15,1.7,427,Cumple,1
3,2025-03-04,Norte,Móvil,62.5,119.4,12,30,38.2,116,Cumple,1
4,2025-03-05,Norte,Empresas,356.1,111.0,9,30,21.5,389,No cumple,0


## 2. Desempeño por región y tipo de servicio

In [3]:
claves = ['region', 'tipo_servicio']
res = resumen_grupos(df, claves).merge(ic_sla_bootstrap(df, claves), on=claves)
res

,region,tipo_servicio,registros,trafico_prom_gb,latencia_prom_ms,incidencias_total,tickets_total,tiempo_resolucion_prom_hr,clientes_afectados_total,pct_sla_cumple,ic95_sla_inf,ic95_sla_sup
0,Centro,Empresas,24,342.85,69.42,343,681,27.95,7359,45.83,25.0,66.7
1,Centro,Internet Hogar,27,272.18,66.95,338,989,21.67,6981,37.04,18.5,59.3
2,Centro,Móvil,17,265.20,67.92,181,487,24.26,4138,52.94,29.4,76.5
3,Norte,Empresas,18,344.59,68.59,229,485,20.57,4649,38.89,16.7,61.1
4,Norte,Internet Hogar,30,294.99,59.56,415,896,22.48,8334,43.33,26.7,60.0
5,Norte,Móvil,17,268.40,69.44,204,600,27.28,4475,52.94,29.4,76.5
6,Sur,Empresas,22,251.93,69.41,303,748,27.03,5323,59.09,36.4,77.3
7,Sur,Internet Hogar,22,240.49,62.92,360,734,30.60,5610,63.64,40.9,81.8
8,Sur,Móvil,23,300.51,53.51,336,658,27.03,6206,43.48,26.1,65.2


In [4]:
pv = res.pivot(index='region', columns='tipo_servicio', values='pct_sla_cumple')
px.imshow(pv, text_auto='.1f', color_continuous_scale='RdYlGn', title='% cumplimiento SLA')

## 3. Relaciones entre variables y pruebas estadísticas

In [5]:
corr = matriz_correlacion(df)
corr

,trafico_datos_gb,latencia_ms,incidencias_reportadas,tickets_soporte,tiempo_resolucion_hr,clientes_afectados,sla_cumple
trafico_datos_gb,1.000,-0.077,-0.123,-0.082,-0.210,-0.040,-0.026
latencia_ms,-0.077,1.000,0.006,-0.203,-0.068,-0.034,0.092
incidencias_reportadas,-0.123,0.006,1.000,-0.075,-0.000,0.064,-0.092
tickets_soporte,-0.082,-0.203,-0.075,1.000,0.067,-0.009,-0.071
tiempo_resolucion_hr,-0.210,-0.068,-0.000,0.067,1.000,0.123,-0.040
clientes_afectados,-0.040,-0.034,0.064,-0.009,0.123,1.000,-0.024
sla_cumple,-0.026,0.092,-0.092,-0.071,-0.040,-0.024,1.000


In [6]:
pd.concat([comparar_grupos(df, 'region'), comparar_grupos(df, 'tipo_servicio')])

,factor,variable,H,p_valor
0,region,latencia_ms,1.094,0.579
1,region,tiempo_resolucion_hr,4.245,0.120
2,region,tickets_soporte,0.331,0.848
3,region,clientes_afectados,0.423,0.809
0,tipo_servicio,latencia_ms,1.885,0.390
1,tipo_servicio,tiempo_resolucion_hr,0.402,0.818
2,tipo_servicio,tickets_soporte,1.511,0.470
3,tipo_servicio,clientes_afectados,0.131,0.937


In [7]:
pd.DataFrame([independencia_sla(df, 'region'), independencia_sla(df, 'tipo_servicio')])

,factor,chi2,p_valor
0,region,2.110,0.348
1,tipo_servicio,0.077,0.962


## 4. Dashboard y archivos de salida

In [8]:
salida = RAIZ / 'output'
salida.mkdir(exist_ok=True)
guardar_dashboard(df, res, corr, salida / 'dashboard.html')
df[['fecha_registro','region','tipo_servicio','incidencias_reportadas','tickets_soporte','tiempo_resolucion_hr']].sort_values('fecha_registro').to_csv(salida / 'soporte_tecnico.csv', index=False, encoding='utf-8-sig')
df[['fecha_registro','tipo_servicio','clientes_afectados','latencia_ms','cumplimiento_sla']].sort_values('fecha_registro').to_csv(salida / 'experiencia_clientes.csv', index=False, encoding='utf-8-sig')
res.to_csv(salida / 'resumen_region_servicio.csv', index=False, encoding='utf-8-sig')
corr.to_csv(salida / 'correlaciones.csv', encoding='utf-8-sig')
print('Salidas actualizadas en', salida)

Salidas actualizadas en C:\Users\eddy_\Documents\IACC\Ciencia de datos\Trabajos\Semana 8\output


## 5. Conclusiones

- El cumplimiento global de SLA es 48 %. Sur y la combinación Sur-Internet Hogar muestran los mejores valores, pero las pruebas de Kruskal-Wallis y chi-cuadrado no detectan diferencias significativas entre regiones ni servicios, y los intervalos de confianza de los grupos se traslapan.
- Las correlaciones entre variables son débiles; el SLA no se asocia con la latencia ni con los tiempos de resolución.